# Modelo Bag-of-Words (BoW)

**Curso:** Procesamiento de Lenguaje Natural · Unidad 2 — Semántica distribucional y representación de texto  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 6 — Vector Semantics  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

---

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. Explicar la representación **Bag-of-Words** (bolsa de palabras) y sus supuestos principales
2. Construir un vocabulario manualmente con el **TweetTokenizer** de NLTK
3. Construir una matriz de características BoW con el **CountVectorizer** de scikit-learn
4. Interpretar una matriz documento–término y razonar sobre su dispersión (*sparsity*)
5. Identificar las limitaciones de BoW como motivación para TF-IDF y los *word embeddings*

## ¿Qué es el modelo Bag-of-Words?

BoW representa un documento como un **multiconjunto no ordenado de ocurrencias de palabras**: un vector de conteos de palabras sobre un vocabulario fijo. Se descartan el orden de las palabras y la gramática.

Dado el vocabulario V = {w₁, w₂, …, wₙ}, el documento dᵢ se convierte en:

```
dᵢ = [count(w₁, dᵢ),  count(w₂, dᵢ),  …,  count(wₙ, dᵢ)]
```

Esto produce una **matriz documento–término** de dimensión (|D| × |V|), normalmente muy dispersa.

## Conjunto de datos

**`dataset_reseñas_espanol.csv`** — 200 reseñas cortas de películas en español (`Reseña`) etiquetadas como *Positiva* / *Negativa*. Ubicado en `data/`.


---

## 1. Configuración del entorno


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import TweetTokenizer

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

DIR_DATA = Path.cwd() / 'data'
print(f'Directorio de datos: {DIR_DATA}')
print(f'¿Existe el directorio?: {DIR_DATA.exists()}')

> **Nota:** `pathlib.Path.cwd()` resuelve las rutas relativas al directorio de trabajo del notebook, sin nombres de usuario fijos ni separadores propios de cada sistema operativo. No es necesario manipular `sys.path`; los paquetes instalados siempre se pueden importar.


---

## 2. Cargar el conjunto de datos


In [ ]:
corpus = pd.read_csv(DIR_DATA / 'dataset_reseñas_espanol.csv', sep=',')

In [ ]:
corpus.info()

In [ ]:
corpus.head(10)

> **Interpretación de la salida:** El conjunto de datos tiene 200 filas y 2 columnas: `Reseña` (texto de la reseña) y `Sentimiento` (etiqueta: *Positiva* / *Negativa*). Ambas son columnas `object` (cadenas) sin valores nulos. La distribución balanceada de clases lo hace adecuado como referencia para clasificación binaria de texto.


---

## 3. Vocabulario manual con TweetTokenizer

Construimos un vocabulario manualmente para entender qué produce la tokenización antes de delegar el trabajo a `CountVectorizer`. El `TweetTokenizer` de NLTK maneja bien textos cortos e informales, emoticones, hashtags y signos de puntuación.

**Nota de implementación:** la pertenencia al vocabulario se comprueba contra un `set`, lo que da una búsqueda promedio O(1) en lugar del costo O(n) de `if word not in list`.


In [ ]:
tknzr = TweetTokenizer()
vocab_set = set()

for row in corpus['Reseña'].tolist():
    for word in tknzr.tokenize(row):
        vocab_set.add(word.lower())

bag = sorted(vocab_set)
print(f'Tamaño del vocabulario: {len(bag)}')
print(f'Primeros 20 tokens: {bag[:20]}')

> **Interpretación de la salida:**
>
> - ~88 tokens únicos en minúsculas a partir de 200 reseñas cortas.
> - Se incluyen **tokens de puntuación** (`.`, `,`): `TweetTokenizer` los conserva. Un *pipeline* de producción los filtraría con una longitud mínima de caracteres o una lista de *stopwords*.
> - `sorted(vocab_set)` garantiza un **orden reproducible** entre ejecuciones, ya que los conjuntos en Python no tienen orden.
> - Usar un `set` evita el costo O(n²) de `if word not in bag` sobre una lista creciente, algo crítico en corpus grandes.


---

## 4. Bag-of-Words con CountVectorizer

`CountVectorizer` construye el vocabulario y la matriz documento–término en una sola llamada a `fit_transform`:
- **`fit`** — recorre todos los documentos y construye un vocabulario interno (ordenado y en minúsculas)
- **`transform`** — convierte cada documento en un vector de conteos sobre ese vocabulario

El resultado es una **`csr_matrix` dispersa** de dimensión `(n_documentos, |V|)`.


In [ ]:
bow = CountVectorizer(analyzer='word', ngram_range=(1, 1))
features = bow.fit_transform(corpus['Reseña'].tolist())

print(f'Dimensión de la matriz de características: {features.shape}')
print(f'Tamaño del vocabulario (CountVectorizer): {len(bow.vocabulary_)}')
print(f'Dispersión: {1 - features.nnz / (features.shape[0] * features.shape[1]):.2%}')

> **Interpretación de la salida:**
>
> - La **dimensión** es `(200, |V|)`: una fila por reseña y una columna por término del vocabulario.
> - El **vocabulario de CountVectorizer** puede diferir del conteo manual porque usa su propio tokenizador basado en expresiones regulares y, por defecto, pasa a minúsculas sin eliminar tildes.
> - La **dispersión** supera el 95 % en documentos cortos: la mayoría de las palabras del vocabulario no aparecen en una reseña dada. Esto motiva la ponderación TF-IDF (reduce el dominio de las palabras comunes) o los *embeddings* densos (capturan similitud semántica).


In [ ]:
# Vista densa — práctica solo para matrices de características pequeñas
features.toarray()

---

## 5. Matriz de características como DataFrame


In [ ]:
df = pd.DataFrame(
    features.toarray(),
    index=[f'#{i}' for i in range(1, 1 + len(corpus))],
    columns=bow.get_feature_names_out(),
)
df.index.name = 'Reseña'
df

> **Interpretación de la salida:**
>
> - Cada **fila** es una reseña; cada **columna** es un término del vocabulario; cada **celda** es el conteo bruto de ocurrencias.
> - La mayoría de las celdas son **cero**, lo que confirma la alta dispersión. En la práctica, conviene almacenarla como matriz dispersa (`features`) en lugar de un `DataFrame` denso.
> - Reseñas idénticas producen vectores de conteo idénticos: BoW no distingue duplicados ni paráfrasis.

## Resumen

| Concepto | Detalle |
|---------|--------|
| Representación | Vector de conteos por documento |
| Orden de las palabras | **Se pierde** — BoW es independiente de la posición |
| Vocabulario | Se construye a partir del corpus de entrenamiento en `fit` |
| Tipo de matriz | Dispersa (`scipy.sparse.csr_matrix`) |
| Limitación principal | Alta dispersión; no captura similitud semántica |
| Siguiente paso | Ponderar los conteos por la frecuencia inversa de documento → **TF-IDF** |
